In [1]:
!pip install pyspark

In [2]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("RealEstateBigData") \
    .getOrCreate()

print("SparkSession creada correctamente")

SparkSession creada correctamente


In [3]:
dim_city = spark.read.csv(
    "dim_city.csv",
    header=True,
    inferSchema=True
)

dim_neighborhood = spark.read.csv(
    "dim_neighborhood.csv",
    header=True,
    inferSchema=True
)

dim_property_type = spark.read.csv(
    "dim_property_type.csv",
    header=True,
    inferSchema=True
)

dim_source = spark.read.csv(
    "dim_source.csv",
    header=True,
    inferSchema=True
)

dim_time = spark.read.csv(
    "dim_time.csv",
    header=True,
    inferSchema=True
)

fact_properties = spark.read.csv(
    "fact_properties.csv",
    header=True,
    inferSchema=True
)

In [4]:
fact_properties.printSchema()

root
 |-- fact_id: integer (nullable = true)
 |-- city_id: integer (nullable = true)
 |-- neighborhood_id: integer (nullable = true)
 |-- property_type_id: integer (nullable = true)
 |-- source_id: integer (nullable = true)
 |-- time_id: integer (nullable = true)
 |-- price_eur: integer (nullable = true)
 |-- area_m2: integer (nullable = true)
 |-- bedrooms: integer (nullable = true)
 |-- bathrooms: integer (nullable = true)
 |-- price_per_m2: double (nullable = true)



In [5]:
fact_properties.show()

+-------+-------+---------------+----------------+---------+-------+---------+-------+--------+---------+------------+
|fact_id|city_id|neighborhood_id|property_type_id|source_id|time_id|price_eur|area_m2|bedrooms|bathrooms|price_per_m2|
+-------+-------+---------------+----------------+---------+-------+---------+-------+--------+---------+------------+
|      1|      1|              1|               1|        1|      1|  1250000|    120|       3|        2|    10416.67|
|      2|      1|              2|               2|        1|      1|  2400000|    210|       5|        4|    11428.57|
|      3|      1|              1|               1|        1|      1|  1800000|    150|       4|        3|     12000.0|
|      4|      1|              3|               3|        1|      1|  3200000|    300|       6|        5|    10666.67|
|      5|      1|              4|               1|        1|      1|   950000|     95|       2|        2|     10000.0|
|      6|      2|              5|               

In [6]:
real_estate_df = (
    fact_properties
    .join(dim_city, "city_id")
    .join(dim_neighborhood, "neighborhood_id")
    .join(dim_property_type, "property_type_id")
    .join(dim_source, "source_id")
    .join(dim_time, "time_id")
)

real_estate_df.show()

+-------+---------+----------------+---------------+-------+-------+---------+-------+--------+---------+------------+------+--------------+------------+-------+-------------+---------+-------------------+----+-------+-----+---+--------+
|time_id|source_id|property_type_id|neighborhood_id|city_id|fact_id|price_eur|area_m2|bedrooms|bathrooms|price_per_m2|  city|       country|neighborhood|city_id|property_type|   source|      scraping_date|year|quarter|month|day|day_name|
+-------+---------+----------------+---------------+-------+-------+---------+-------+--------+---------+------------+------+--------------+------------+-------+-------------+---------+-------------------+----+-------+-----+---+--------+
|      1|        1|               1|              1|      1|      1|  1250000|    120|       3|        2|    10416.67|madrid|         spain|   salamanca|      1|    apartment|idealista|2026-05-21 16:55:36|2026|      2|    5| 21|Thursday|
|      1|        1|               2|            

In [7]:
real_estate_df.groupBy("city") \
    .avg("price_eur") \
    .show()

+------+--------------+
|  city|avg(price_eur)|
+------+--------------+
|london|     5560000.0|
|madrid|     1920000.0|
+------+--------------+



In [8]:
real_estate_df.groupBy("property_type") \
    .avg("price_per_m2") \
    .show()

+-------------+-----------------+
|property_type|avg(price_per_m2)|
+-------------+-----------------+
|    penthouse|        16309.525|
|       duplex|        15333.335|
|    townhouse|          21250.0|
|    apartment|          15400.0|
+-------------+-----------------+



In [9]:
from pyspark.sql.functions import avg

real_estate_df.rollup(
    "city",
    "property_type"
).agg(
    avg("price_eur").alias("avg_price")
).show()

+------+-------------+------------------+
|  city|property_type|         avg_price|
+------+-------------+------------------+
|london|    penthouse|         8900000.0|
|london|    apartment|         3800000.0|
|madrid|    penthouse|         2400000.0|
|london|         NULL|         5560000.0|
|london|    townhouse|         5100000.0|
|madrid|    apartment|1333333.3333333333|
|  NULL|         NULL|         3740000.0|
|madrid|       duplex|         3200000.0|
|london|       duplex|         6200000.0|
|madrid|         NULL|         1920000.0|
+------+-------------+------------------+



In [10]:
real_estate_df.cube(
    "city",
    "property_type"
).agg(
    avg("price_per_m2").alias("avg_price_m2")
).show()

+------+-------------+------------------+
|  city|property_type|      avg_price_m2|
+------+-------------+------------------+
|  NULL|    apartment|           15400.0|
|london|    penthouse|          21190.48|
|london|    apartment|         22291.665|
|madrid|    penthouse|          11428.57|
|london|         NULL|         21404.762|
|london|    townhouse|           21250.0|
|madrid|    apartment|10805.556666666665|
|  NULL|       duplex|         15333.335|
|  NULL|    townhouse|           21250.0|
|  NULL|         NULL|16153.571999999996|
|madrid|       duplex|          10666.67|
|london|       duplex|           20000.0|
|madrid|         NULL|         10902.382|
|  NULL|    penthouse|         16309.525|
+------+-------------+------------------+



In [11]:
real_estate_df.groupBy(
    "neighborhood"
).avg(
    "price_per_m2"
).orderBy(
    "avg(price_per_m2)",
    ascending=False
).show()

+------------+-----------------+
|neighborhood|avg(price_per_m2)|
+------------+-----------------+
|  kensington|         23333.33|
|     chelsea|          21250.0|
|notting hill|          21250.0|
|     mayfair|         21190.48|
| westminster|          20000.0|
|    chamberí|         11428.57|
|   salamanca|        11208.335|
|   chamartín|         10666.67|
|      retiro|          10000.0|
+------------+-----------------+

